# 실습 01 · PyTorch 기초: tensor, shape, autograd

**목표**

1. tensor의 `shape`, `dtype`, `device`를 읽는다.
2. broadcasting과 행렬곱에서 shape가 어떻게 바뀌는지 설명한다.
3. `loss.backward()`가 gradient를 계산하는 과정을 실행한다.
4. 학습률을 바꾸고 loss 곡선의 변화를 해석한다.

## 1. 환경·재현성·device

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", DEVICE)


`device`는 tensor와 model이 계산되는 위치입니다. Colab을 활용한 GPU 실습이 기본이며, GPU가 없는 환경에서도 코드 구조는 모두 같습니다.

## 2. NumPy와 tensor 오가기

pandas·NumPy로 준비한 데이터를 PyTorch 모델에 넣을 때 자주 사용하는 연결입니다. 메모리 공유 여부와 dtype 변화를 함께 확인합니다.

In [ ]:
numpy_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
tensor_view = torch.from_numpy(numpy_array)
tensor_copy = torch.tensor(numpy_array)
numpy_array[0, 0] = 99
print("from_numpy는 메모리 공유:", tensor_view)
print("torch.tensor는 복사:", tensor_copy)
print("dtype:", tensor_view.dtype, "shape:", tensor_view.shape)


## 2. tensor 생성: 값·dtype·shape

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
labels = torch.tensor([0, 1])
print("x =\n", x)
print("x.shape:", x.shape, "x.dtype:", x.dtype)
print("labels.shape:", labels.shape, "labels.dtype:", labels.dtype)


**Shape checkpoint ①**

- `x`: 샘플 2개 × feature 3개 → `(2, 3)`
- `labels`: 샘플마다 정답 1개 → `(2,)`
- **분류 (Classification)**: 입력 logit `(batch, classes)`와 정답 `(batch,)`를 받습니다. 예: 3개 클래스 분류 시 logit은 `(batch, 3)`
- **회귀 분석 (Regression)**: 연속값을 예측하므로 logit `(batch, 1)` 또는 `(batch, out_features)`와 정답 `(batch, 1)`을 받습니다. 예: 주택 가격 예측은 `(batch, 1)`

## 3. 축 변환과 reshape

In [ ]:
image_batch = torch.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5)
print("image batch (N,C,H,W):", image_batch.shape)
flattened = image_batch.flatten(start_dim=1)
print("flattened (N,features):", flattened.shape)
restored = flattened.reshape(2, 3, 4, 5)
print("restored:", restored.shape)


`reshape`는 원소 수를 보존합니다. 이미지 모델의 `(N,C,H,W)`와 MLP의 `(N,features)`가 연결되는 지점입니다.

## tensor 결합: `cat`과 `stack`

`cat`은 기존 축을 이어 붙이고, `stack`은 새 축을 만듭니다. mini-batch 축을 만들 때 두 연산의 차이가 중요합니다.

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])
print("cat   :", torch.cat([a, b]).shape, torch.cat([a, b]))
print("stack :", torch.stack([a, b]).shape, torch.stack([a, b]))
print("stack dim=1:", torch.stack([a, b], dim=1).shape)


## 축을 지정하는 reduction

평균을 어느 축으로 계산하느냐에 따라 “샘플별 요약”과 “feature별 요약”이 달라집니다.

In [ ]:
batch = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("batch:", batch.shape)
print("전체 평균:", batch.mean().shape, batch.mean().item())
print("샘플별 평균:", batch.mean(dim=1).shape, batch.mean(dim=1))
print("feature별 평균:", batch.mean(dim=0).shape, batch.mean(dim=0))

## 4. indexing과 broadcasting

In [ ]:
first_sample = x[0]
feature_mean = x.mean(dim=0)
centered = x - feature_mean  # (2,3) - (3,) broadcasting
print("first_sample:", first_sample, first_sample.shape)
print("feature_mean:", feature_mean, feature_mean.shape)
print("centered shape:", centered.shape)
print("centered mean:", centered.mean(dim=0))


**예상 질문:** `(2,3)`에서 `(3,)`을 빼는데 왜 오류가 나지 않을까요?  
마지막 축의 크기가 같아 작은 tensor가 각 샘플에 반복 적용되기 때문입니다.

## 5. 행렬곱으로 만드는 간단한 forward

In [ ]:
X = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
W = torch.tensor([[0.2, -0.1], [0.5, 0.3], [-0.4, 0.7]])
b = torch.tensor([0.1, -0.2])
logits = X @ W + b
print("X:", X.shape, "W:", W.shape, "b:", b.shape)
print("logits:", logits.shape)
print(logits)


**Shape checkpoint ②:** `(2,3) @ (3,2) + (2,) → (2,2)`  
여기서 마지막 2는 class 수라고 해석할 수 있습니다.

## `nn.Linear`로 같은 계산 표현하기

`nn.Linear(in_features, out_features)`는 `X @ Wᵀ + b`를 수행하며, 파라미터를 자동으로 등록합니다.

In [ ]:
linear = torch.nn.Linear(3, 2)
with torch.no_grad():
    linear.weight.copy_(W.T)
    linear.bias.copy_(b)
module_logits = linear(X)
print("manual == module:", torch.allclose(logits, module_logits))
print("registered parameters:", [(name, tuple(value.shape)) for name, value in linear.named_parameters()])


## 6. autograd: 계산 그래프와 gradient

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
x_one = torch.tensor(2.0)
y_true = torch.tensor(8.0)

y_pred = w * x_one + b
loss = (y_pred - y_true) ** 2
loss.backward()

print("prediction:", y_pred.item(), "loss:", loss.item())
print("d(loss)/dw:", w.grad.item())
print("d(loss)/db:", b.grad.item())


`requires_grad=True`인 tensor가 참여한 연산을 PyTorch가 기록하고, `backward()`가 chain rule로 미분합니다.

gradient는 **파라미터를 어느 방향으로 얼마나 바꿀지 알려 주는 값**이지, 업데이트 자체는 아닙니다.

## `detach`와 `no_grad`

예측값을 기록하거나 파라미터를 직접 갱신할 때는 새 계산 그래프가 필요하지 않습니다.

In [ ]:
tracked = w * x_one + b
detached = tracked.detach()
with torch.no_grad():
    untracked = w * x_one + b
print("tracked requires_grad:", tracked.requires_grad)
print("detached requires_grad:", detached.requires_grad)
print("no_grad result:", untracked.requires_grad)


## 7. gradient 누적과 `zero_grad`

In [ ]:
w.grad.zero_(); b.grad.zero_()
loss_a = (w * x_one + b - y_true) ** 2
loss_a.backward()
first_grad = w.grad.item()

loss_b = (w * x_one + b - y_true) ** 2  # 새 계산 그래프
loss_b.backward()
print("한 번의 gradient:", first_grad)
print("두 번 누적된 gradient:", w.grad.item())

PyTorch gradient는 기본적으로 누적됩니다.  
새 계산 그래프를 두 번 미분하면 `w.grad`가 두 번 더해지는 것을 확인할 수 있습니다.

In [ ]:
w.grad.zero_(); b.grad.zero_()
for _ in range(2):
    current_loss = (w * x_one + b - y_true) ** 2
    current_loss.backward()
print("두 번 누적된 w.grad:", w.grad.item())
w.grad.zero_(); b.grad.zero_()
print("초기화 후:", w.grad.item(), b.grad.item())


> 학습 루프에서는 보통 매 iteration마다 `optimizer.zero_grad()`를 먼저 호출합니다.

## 8. 수동 gradient descent

In [ ]:
torch.manual_seed(SEED)
x_train = torch.linspace(-2, 2, 80, device=DEVICE).reshape(-1, 1)
noise = 0.25 * torch.randn_like(x_train)
y_train = 3.0 * x_train + 2.0 + noise

weight = torch.zeros(1, requires_grad=True, device=DEVICE)
bias = torch.zeros(1, requires_grad=True, device=DEVICE)
LEARNING_RATE = 0.08  # 핵심 변경 변수
losses = []

for step in range(40):
    prediction = x_train * weight + bias
    train_loss = ((prediction - y_train) ** 2).mean()
    train_loss.backward()
    with torch.no_grad():
        weight -= LEARNING_RATE * weight.grad
        bias -= LEARNING_RATE * bias.grad
    weight.grad.zero_(); bias.grad.zero_()
    losses.append(train_loss.item())

print(f"weight={weight.item():.3f}, bias={bias.item():.3f}, final loss={losses[-1]:.4f}")


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(losses, color="#009E96", linewidth=2)
plt.xlabel("step"); plt.ylabel("MSE loss"); plt.title("Manual gradient descent")
plt.grid(alpha=.2); plt.show()


## optimizer로 같은 회귀 문제 학습하기

실제 모델에서는 직접 파라미터를 빼는 대신 optimizer가 업데이트 규칙을 관리합니다.

## 여러 Optimizer 비교: SGD vs Adam

다양한 optimizer를 같은 회귀 문제에 적용하고, loss 감소 곡선을 비교해봅시다.  
**학습률**과 **optimizer 종류**에 따라 수렴 속도와 최종 loss가 어떻게 달라지는지 관찰하는 것이 목표입니다.

In [ ]:
# 다양한 optimizer 학습 곡선 비교
optimizers_to_test = {
    'SGD (lr=0.08)': ('sgd', 0.08),
    'SGD (lr=0.02)': ('sgd', 0.02),
    'Adam (lr=0.001)': ('adam', 0.01),
}

optimizer_results = {}

for opt_name, (opt_type, lr) in optimizers_to_test.items():
    torch.manual_seed(SEED)
    model = torch.nn.Linear(1, 1).to(DEVICE)
    
    # optimizer 생성
    if opt_type == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif opt_type == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif opt_type == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    criterion = torch.nn.MSELoss()
    losses = []
    
    for step in range(40):
        optimizer.zero_grad()
        pred = model(x_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    optimizer_results[opt_name] = {
        'losses': losses,
        'weight': model.weight.item(),
        'bias': model.bias.item(),
        'final_loss': losses[-1]
    }
    print(f"{opt_name:20s} | weight={model.weight.item():7.3f}, bias={model.bias.item():7.3f}, loss={losses[-1]:.4f}")

print("\n✓ 목표값: weight=3.000, bias=2.000")

In [ ]:
# optimizer 비교: 전체 곡선과 조기 수렴 구간
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = {'SGD (lr=0.08)': '#FF6B6B', 'SGD (lr=0.02)': '#FFA500', 
          'Adam (lr=0.001)': '#009E96'}

# 왼쪽: 전체 곡선 (로그 스케일)
for opt_name, result in optimizer_results.items():
    axes[0].plot(result['losses'], marker='o', markersize=3, linewidth=2.5, 
                 label=opt_name, alpha=0.8, color=colors.get(opt_name, '#333'))

axes[0].set_xlabel('Step', fontsize=11, fontweight='bold')
axes[0].set_ylabel('MSE Loss (log scale)', fontsize=11, fontweight='bold')
axes[0].set_title('Learning Curves - All Steps', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_yscale('log')

# 오른쪽: 초기 10 step (선형 스케일)
for opt_name, result in optimizer_results.items():
    axes[1].plot(result['losses'][:10], marker='s', markersize=6, linewidth=2.5, 
                 label=opt_name, alpha=0.8, color=colors.get(opt_name, '#333'))

axes[1].set_xlabel('Step', fontsize=11, fontweight='bold')
axes[1].set_ylabel('MSE Loss', fontsize=11, fontweight='bold')
axes[1].set_title('Early Convergence - First 10 Steps', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## mini-batch를 만드는 Dataset과 DataLoader

전체 데이터를 작은 batch로 나누면 메모리를 절약하고 여러 번 파라미터를 업데이트할 수 있습니다. Mini-batch 경사하강법의 기본

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
regression_dataset = TensorDataset(x_train.cpu(), y_train.cpu())
regression_loader = DataLoader(regression_dataset, batch_size=16, shuffle=True)
for batch_id, (xb, yb) in enumerate(regression_loader):
    print(f"batch {batch_id}: X{tuple(xb.shape)} y{tuple(yb.shape)}")
    if batch_id == 2:
        break


## 🎯 학습자 변경 과제: Optimizer와 Learning Rate 실험

위의 optimizer 비교 코드(4개 optimizer)에서 **하나 이상의 학습률을 변경**하고 다시 실행해보세요.

### 실험 아이디어:
1. **SGD의 학습률을 극단값으로 설정**: `0.001` (매우 작음) 또는 `0.5` (매우 큼)
2. **Adam의 학습률 변경**: `0.001`, `0.05`, `0.1` 등으로 조정
3. **새로운 optimizer 추가**: `torch.optim.RMSprop`, `torch.optim.LAMB` 등

### 관찰할 사항:
- 초기 10 step에서 어느 optimizer가 가장 가파른 loss 감소를 보이나?
- 최종 loss (40 step 후)는 어느 것이 가장 낮나?
- 학습률이 너무 크거나 작으면 어떤 현상이 나타나나?
- 다른 optimizer들 사이에서 노이즈/진동의 정도가 다른가?

### 기록 양식:

**내 실험 결과:**
```
변경 사항: [어떤 학습률/optimizer를 변경했는지 작성]

관찰:
- 가장 빠른 수렴: [어느 optimizer]
- 가장 낮은 최종 loss: [어느 optimizer]  
- 가장 안정적인 곡선: [어느 optimizer]
- 불안정한 곡선: [어느 optimizer]

해석:
[왜 그런 결과가 나왔는지 설명. learning rate의 영향]
```